# Phase 2: Stochastic Modelling & Derivatives Pricing  
## Notebook 02_07 — Exotic Options Pricing with Monte Carlo

### Research question

How can Monte Carlo simulation price path-dependent exotic options such as Asian, barrier, and lookback options, and how do discretisation, monitoring frequency, and variance reduction affect the result?

This notebook extends `02_06_monte_carlo_gbm_and_fat_tails`.

The previous notebook used Monte Carlo mainly for terminal European payoffs. This notebook moves to payoffs that depend on the whole path:

1. **Asian options** — depend on the average price;
2. **Barrier options** — depend on whether a barrier is crossed;
3. **Lookback options** — depend on the running maximum or minimum;
4. **Monte Carlo confidence intervals**;
5. **Antithetic variates**;
6. **Control variates**;
7. **Monitoring-frequency bias**;
8. **Brownian-bridge correction for barrier crossing**;
9. **Pathwise diagnostics and saved pricing reports**.

The key idea is:

> Monte Carlo becomes especially valuable when the payoff depends on the realised path rather than only the terminal price.

## 1. Why exotic options need simulation

A European call payoff is:

$$
(S_T-K)^+
$$
It depends only on the terminal stock price.

Many exotic options depend on the full path:

### Asian call

$$
\left(\frac{1}{N}\sum_{i=1}^{N}S_{t_i}-K\right)^+
$$
### Up-and-out call

$$
(S_T-K)^+ \mathbf 1_{\{\max_i S_{t_i}<B\}}
$$
### Floating-strike lookback call

$$
S_T-\min_i S_{t_i}
$$
For these payoffs, a terminal simulation is not enough. We need simulated paths:

$$
S_{t_0},S_{t_1},\dots,S_{t_N}
$$
The price is estimated as:

$$
\begin{aligned}
V_0 &= e^{-rT} \mathbb E^{\mathbb Q} [ g(S_{t_0},S_{t_1},\dots,S_T) ]
\end{aligned}
$$

## 2. Model setup: risk-neutral GBM

We simulate paths under risk-neutral geometric Brownian motion:

$$
dS_t=(r-q)S_tdt+\sigma S_tdW_t^{\mathbb Q}
$$
Using exact discrete transitions:

$$
\begin{aligned}
S_{t+\Delta t} &= S_t \exp \Bigg[ (r-q-\frac{1}{2}\sigma^2)\Delta t \\
&\quad + \sigma\sqrt{\Delta t}Z \Bigg]
\end{aligned}
$$
where:

$$
Z\sim\mathcal N(0,1)
$$
This avoids Euler discretisation error for the GBM transition itself.

However, path-dependent payoffs can still suffer from **monitoring bias**. For example, a barrier may be crossed between two simulated time points but missed by the discrete grid.

## 3. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from math import erf, exp, log, sqrt
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=42)

In [ ]:
@dataclass(frozen=True)
class ExoticMCConfig:
    """
    Monte Carlo configuration for exotic option pricing.
    """
    s0: float = 100.0
    strike: float = 100.0
    maturity: float = 1.0
    risk_free_rate: float = 0.04
    dividend_yield: float = 0.00
    volatility: float = 0.20
    n_paths: int = 120_000
    n_steps: int = 252
    seed: int = 42


config = ExoticMCConfig()
config

## 4. Black-Scholes-Merton reference functions

We use vanilla European call and put prices as validation references.

If a Monte Carlo engine cannot price a vanilla option correctly, it should not be trusted for exotics.

In [ ]:
def normal_cdf(x: float | np.ndarray) -> float | np.ndarray:
    """
    Standard normal CDF.
    """
    x = np.asarray(x)
    return 0.5 * (1.0 + np.vectorize(erf)(x / np.sqrt(2.0)))


def bsm_price(
    spot: float,
    strike: float,
    maturity: float,
    risk_free_rate: float,
    dividend_yield: float,
    volatility: float,
    option_type: str
) -> float:
    """
    Black-Scholes-Merton European option price.
    """
    if maturity <= 0:
        if option_type == "call":
            return max(spot - strike, 0.0)
        if option_type == "put":
            return max(strike - spot, 0.0)
        raise ValueError("option_type must be 'call' or 'put'.")

    d1 = (
        log(spot / strike)
        + (risk_free_rate - dividend_yield + 0.5 * volatility ** 2) * maturity
    ) / (volatility * sqrt(maturity))

    d2 = d1 - volatility * sqrt(maturity)

    discounted_spot = spot * exp(-dividend_yield * maturity)
    discounted_strike = strike * exp(-risk_free_rate * maturity)

    if option_type == "call":
        return float(discounted_spot * normal_cdf(d1) - discounted_strike * normal_cdf(d2))

    if option_type == "put":
        return float(discounted_strike * normal_cdf(-d2) - discounted_spot * normal_cdf(-d1))

    raise ValueError("option_type must be 'call' or 'put'.")

In [ ]:
bsm_call_reference = bsm_price(
    spot=config.s0,
    strike=config.strike,
    maturity=config.maturity,
    risk_free_rate=config.risk_free_rate,
    dividend_yield=config.dividend_yield,
    volatility=config.volatility,
    option_type="call"
)

bsm_put_reference = bsm_price(
    spot=config.s0,
    strike=config.strike,
    maturity=config.maturity,
    risk_free_rate=config.risk_free_rate,
    dividend_yield=config.dividend_yield,
    volatility=config.volatility,
    option_type="put"
)

pd.Series({
    "bsm_call_reference": bsm_call_reference,
    "bsm_put_reference": bsm_put_reference
})

## 5. GBM path simulator

The simulator supports ordinary Monte Carlo and antithetic variates.

Antithetic variates pair each normal shock matrix $Z$ with $-Z$, often reducing variance for monotonic payoffs.

In [ ]:
def simulate_gbm_paths(
    config: ExoticMCConfig,
    n_paths: int | None = None,
    n_steps: int | None = None,
    seed: int | None = None,
    antithetic: bool = False
) -> tuple[np.ndarray, np.ndarray]:
    """
    Simulate risk-neutral GBM paths with exact discrete transitions.

    Returns
    -------
    time_grid:
        Array of time points.

    paths:
        Simulated price paths with shape (n_paths, n_steps + 1).
    """
    if n_paths is None:
        n_paths = config.n_paths

    if n_steps is None:
        n_steps = config.n_steps

    if seed is None:
        seed = config.seed

    local_rng = np.random.default_rng(seed)

    dt = config.maturity / n_steps
    time_grid = np.linspace(0.0, config.maturity, n_steps + 1)

    if antithetic:
        half_paths = n_paths // 2
        z_half = local_rng.standard_normal(size=(half_paths, n_steps))
        z = np.vstack([z_half, -z_half])
        if z.shape[0] < n_paths:
            extra = local_rng.standard_normal(size=(1, n_steps))
            z = np.vstack([z, extra])
    else:
        z = local_rng.standard_normal(size=(n_paths, n_steps))

    log_increments = (
        (config.risk_free_rate - config.dividend_yield - 0.5 * config.volatility ** 2) * dt
        + config.volatility * sqrt(dt) * z
    )

    log_paths = np.zeros((n_paths, n_steps + 1), dtype=float)
    log_paths[:, 0] = log(config.s0)
    log_paths[:, 1:] = log(config.s0) + np.cumsum(log_increments, axis=1)

    paths = np.exp(log_paths)

    return time_grid, paths

In [ ]:
time_grid, paths = simulate_gbm_paths(
    config=config,
    n_paths=5_000,
    n_steps=config.n_steps,
    seed=123
)

paths.shape

In [ ]:
plt.figure(figsize=(12, 6))

for i in range(50):
    plt.plot(time_grid, paths[i], alpha=0.35)

plt.title("Risk-Neutral GBM Sample Paths")
plt.xlabel("Time")
plt.ylabel("Price")
plt.show()

## 6. Generic Monte Carlo pricing helper

The pricing helper converts simulated payoffs into:

- discounted price;
- standard error;
- 95% confidence interval;
- number of paths.

This structure will be reused across all exotic payoffs.

In [ ]:
def monte_carlo_price_from_payoff(
    payoff: np.ndarray,
    maturity: float,
    risk_free_rate: float,
    label: str
) -> dict:
    """
    Convert simulated payoffs into a Monte Carlo price and confidence interval.
    """
    discounted_payoff = np.exp(-risk_free_rate * maturity) * payoff

    price = float(np.mean(discounted_payoff))
    standard_error = float(np.std(discounted_payoff, ddof=1) / np.sqrt(len(discounted_payoff)))

    return {
        "instrument": label,
        "price": price,
        "standard_error": standard_error,
        "lower_95": price - 1.96 * standard_error,
        "upper_95": price + 1.96 * standard_error,
        "n_paths": int(len(payoff))
    }

## 7. Vanilla option validation

Before pricing exotics, validate the path engine by pricing vanilla call and put options from terminal prices.

The Monte Carlo estimates should be close to the Black-Scholes-Merton references.

In [ ]:
def vanilla_payoff(paths: np.ndarray, strike: float, option_type: str) -> np.ndarray:
    """
    Vanilla call/put payoff from terminal path values.
    """
    terminal = paths[:, -1]

    if option_type == "call":
        return np.maximum(terminal - strike, 0.0)

    if option_type == "put":
        return np.maximum(strike - terminal, 0.0)

    raise ValueError("option_type must be 'call' or 'put'.")


_, validation_paths = simulate_gbm_paths(
    config=config,
    n_paths=config.n_paths,
    n_steps=config.n_steps,
    seed=111,
    antithetic=False
)

vanilla_call_mc = monte_carlo_price_from_payoff(
    payoff=vanilla_payoff(validation_paths, config.strike, "call"),
    maturity=config.maturity,
    risk_free_rate=config.risk_free_rate,
    label="vanilla_call"
)

vanilla_put_mc = monte_carlo_price_from_payoff(
    payoff=vanilla_payoff(validation_paths, config.strike, "put"),
    maturity=config.maturity,
    risk_free_rate=config.risk_free_rate,
    label="vanilla_put"
)

pd.DataFrame([
    {**vanilla_call_mc, "bsm_reference": bsm_call_reference, "error_vs_bsm": vanilla_call_mc["price"] - bsm_call_reference},
    {**vanilla_put_mc, "bsm_reference": bsm_put_reference, "error_vs_bsm": vanilla_put_mc["price"] - bsm_put_reference},
])

## 8. Asian options

An arithmetic Asian call payoff is:

$$
\left(\frac{1}{N}\sum_{i=1}^{N}S_{t_i}-K\right)^+
$$
An arithmetic Asian option usually has no simple Black-Scholes closed form.

A geometric Asian call uses the geometric average:

$$
\begin{aligned}
G &= \exp \left( \frac{1}{N}\sum_{i=1}^{N}\log S_{t_i} \right)
\end{aligned}
$$
with payoff:

$$
(G-K)^+
$$
The geometric Asian option has a closed form under GBM and can be used as a control variate for the arithmetic Asian option.

In [ ]:
def asian_payoffs(
    paths: np.ndarray,
    strike: float,
    option_type: str = "call",
    include_initial: bool = False
) -> pd.DataFrame:
    """
    Compute arithmetic and geometric Asian option payoffs.
    """
    sample = paths if include_initial else paths[:, 1:]

    arithmetic_average = sample.mean(axis=1)
    geometric_average = np.exp(np.log(sample).mean(axis=1))

    if option_type == "call":
        arithmetic_payoff = np.maximum(arithmetic_average - strike, 0.0)
        geometric_payoff = np.maximum(geometric_average - strike, 0.0)
    elif option_type == "put":
        arithmetic_payoff = np.maximum(strike - arithmetic_average, 0.0)
        geometric_payoff = np.maximum(strike - geometric_average, 0.0)
    else:
        raise ValueError("option_type must be 'call' or 'put'.")

    return pd.DataFrame({
        "arithmetic_average": arithmetic_average,
        "geometric_average": geometric_average,
        "arithmetic_payoff": arithmetic_payoff,
        "geometric_payoff": geometric_payoff
    })


_, asian_paths = simulate_gbm_paths(
    config=config,
    n_paths=config.n_paths,
    n_steps=config.n_steps,
    seed=222,
    antithetic=True
)

asian_df = asian_payoffs(asian_paths, config.strike, option_type="call")

asian_arithmetic_mc = monte_carlo_price_from_payoff(
    payoff=asian_df["arithmetic_payoff"].to_numpy(),
    maturity=config.maturity,
    risk_free_rate=config.risk_free_rate,
    label="arithmetic_asian_call"
)

asian_geometric_mc = monte_carlo_price_from_payoff(
    payoff=asian_df["geometric_payoff"].to_numpy(),
    maturity=config.maturity,
    risk_free_rate=config.risk_free_rate,
    label="geometric_asian_call"
)

pd.DataFrame([asian_arithmetic_mc, asian_geometric_mc])

## 9. Geometric Asian closed-form benchmark

For equally spaced monitoring times $t_i$, the log geometric average is normally distributed under GBM.

Let:

$$
G = \exp\left(\frac{1}{N}\sum_{i=1}^{N}\log S_{t_i}\right)
$$
Then:

$$
\log G \sim \mathcal N(m_G, v_G)
$$
where, for monitoring times $t_i$:

$$
\begin{aligned}
m_G &= \log S_0 \\
&\quad + (r-q-\frac{1}{2}\sigma^2) \frac{1}{N}\sum_{i=1}^{N}t_i
\end{aligned}
$$
$$
\begin{aligned}
v_G &= \frac{\sigma^2}{N^2} \sum_{i=1}^{N}\sum_{j=1}^{N}\min(t_i,t_j)
\end{aligned}
$$
Then a geometric Asian call can be priced as a lognormal option on $G$.

In [ ]:
def geometric_asian_closed_form(
    config: ExoticMCConfig,
    option_type: str = "call",
    n_monitoring: int | None = None,
    include_initial: bool = False
) -> float:
    """
    Closed-form price for a discretely monitored geometric Asian option under GBM.
    """
    if n_monitoring is None:
        n_monitoring = config.n_steps

    if include_initial:
        times = np.linspace(0.0, config.maturity, n_monitoring + 1)
    else:
        times = np.linspace(config.maturity / n_monitoring, config.maturity, n_monitoring)

    N = len(times)

    mean_time = np.mean(times)

    # Sum min(t_i, t_j)
    min_matrix = np.minimum.outer(times, times)
    variance_log_g = config.volatility ** 2 * min_matrix.sum() / (N ** 2)

    mean_log_g = (
        log(config.s0)
        + (config.risk_free_rate - config.dividend_yield - 0.5 * config.volatility ** 2) * mean_time
    )

    std_log_g = sqrt(variance_log_g)

    d2 = (mean_log_g - log(config.strike)) / std_log_g
    d1 = d2 + std_log_g

    discount = exp(-config.risk_free_rate * config.maturity)
    expected_g_indicator = exp(mean_log_g + 0.5 * variance_log_g) * normal_cdf(d1)

    if option_type == "call":
        price = discount * (
            expected_g_indicator
            - config.strike * normal_cdf(d2)
        )
    elif option_type == "put":
        price = discount * (
            config.strike * normal_cdf(-d2)
            - exp(mean_log_g + 0.5 * variance_log_g) * normal_cdf(-d1)
        )
    else:
        raise ValueError("option_type must be 'call' or 'put'.")

    return float(price)


geometric_asian_cf = geometric_asian_closed_form(
    config=config,
    option_type="call",
    n_monitoring=config.n_steps,
    include_initial=False
)

pd.Series({
    "geometric_asian_closed_form": geometric_asian_cf,
    "geometric_asian_mc": asian_geometric_mc["price"],
    "mc_error": asian_geometric_mc["price"] - geometric_asian_cf
})

## 10. Control variate for arithmetic Asian option

The arithmetic Asian option is harder to price, but the geometric Asian option is highly correlated with it and has a closed-form price.

Let:

$$
X = \text{discounted arithmetic payoff}
$$
$$
Y = \text{discounted geometric payoff}
$$
and suppose:

$$
\mathbb E[Y]
$$
is known from the closed-form geometric Asian price.

Then:

$$
X^{CV}=X-b(Y-\mathbb E[Y])
$$
with:

$$
b^\star=\frac{\operatorname{Cov}(X,Y)}{\operatorname{Var}(Y)}
$$
This often gives a large variance reduction.

In [ ]:
def arithmetic_asian_control_variate_price(
    asian_df: pd.DataFrame,
    geometric_closed_form_price: float,
    maturity: float,
    risk_free_rate: float
) -> dict:
    """
    Price arithmetic Asian call using geometric Asian control variate.
    """
    discount = exp(-risk_free_rate * maturity)

    X = discount * asian_df["arithmetic_payoff"].to_numpy()
    Y = discount * asian_df["geometric_payoff"].to_numpy()

    b_star = np.cov(X, Y, ddof=1)[0, 1] / np.var(Y, ddof=1)

    adjusted = X - b_star * (Y - geometric_closed_form_price)

    plain_price = float(np.mean(X))
    plain_se = float(np.std(X, ddof=1) / sqrt(len(X)))

    cv_price = float(np.mean(adjusted))
    cv_se = float(np.std(adjusted, ddof=1) / sqrt(len(adjusted)))

    return {
        "plain_arithmetic_asian_price": plain_price,
        "plain_standard_error": plain_se,
        "control_variate_price": cv_price,
        "control_variate_standard_error": cv_se,
        "b_star": float(b_star),
        "variance_reduction_ratio": float((plain_se ** 2) / (cv_se ** 2))
    }


asian_cv_result = arithmetic_asian_control_variate_price(
    asian_df=asian_df,
    geometric_closed_form_price=geometric_asian_cf,
    maturity=config.maturity,
    risk_free_rate=config.risk_free_rate
)

pd.Series(asian_cv_result)

## 11. Barrier options

A barrier option is activated or deactivated when the underlying crosses a barrier.

An up-and-out call payoff is:

$$
(S_T-K)^+
\mathbf 1_{\{\max_t S_t < B\}}
$$
where $B>S_0$.

A down-and-out put payoff is:

$$
(K-S_T)^+
\mathbf 1_{\{\min_t S_t > B\}}
$$
where $B<S_0$.

The challenge is monitoring: if we only check a discrete grid, we can miss barrier crossings between time steps.

In [ ]:
def barrier_payoff_discrete(
    paths: np.ndarray,
    strike: float,
    barrier: float,
    option_type: str,
    barrier_type: str
) -> pd.DataFrame:
    """
    Compute discretely monitored barrier option payoff.
    """
    terminal = paths[:, -1]
    path_max = paths.max(axis=1)
    path_min = paths.min(axis=1)

    if option_type == "call":
        vanilla = np.maximum(terminal - strike, 0.0)
    elif option_type == "put":
        vanilla = np.maximum(strike - terminal, 0.0)
    else:
        raise ValueError("option_type must be 'call' or 'put'.")

    if barrier_type == "up-and-out":
        knocked_out = path_max >= barrier
    elif barrier_type == "down-and-out":
        knocked_out = path_min <= barrier
    elif barrier_type == "up-and-in":
        knocked_out = path_max < barrier
    elif barrier_type == "down-and-in":
        knocked_out = path_min > barrier
    else:
        raise ValueError("Unsupported barrier_type.")

    payoff = np.where(knocked_out, 0.0, vanilla)

    return pd.DataFrame({
        "terminal": terminal,
        "path_max": path_max,
        "path_min": path_min,
        "vanilla_payoff": vanilla,
        "knocked_out_or_not_activated": knocked_out,
        "barrier_payoff": payoff
    })


barrier = 125.0

_, barrier_paths = simulate_gbm_paths(
    config=config,
    n_paths=config.n_paths,
    n_steps=config.n_steps,
    seed=333,
    antithetic=True
)

barrier_df = barrier_payoff_discrete(
    paths=barrier_paths,
    strike=config.strike,
    barrier=barrier,
    option_type="call",
    barrier_type="up-and-out"
)

up_and_out_call_discrete = monte_carlo_price_from_payoff(
    payoff=barrier_df["barrier_payoff"].to_numpy(),
    maturity=config.maturity,
    risk_free_rate=config.risk_free_rate,
    label="up_and_out_call_discrete"
)

pd.Series(up_and_out_call_discrete)

In [ ]:
pd.Series({
    "barrier": barrier,
    "knockout_rate_discrete": barrier_df["knocked_out_or_not_activated"].mean(),
    "vanilla_call_mc_same_paths": monte_carlo_price_from_payoff(
        payoff=barrier_df["vanilla_payoff"].to_numpy(),
        maturity=config.maturity,
        risk_free_rate=config.risk_free_rate,
        label="vanilla_call_same_paths"
    )["price"],
    "up_and_out_call_discrete": up_and_out_call_discrete["price"]
})

## 12. Monitoring-frequency bias

A continuously monitored barrier can be crossed between simulated time points.

If we only monitor daily, weekly, or monthly points, we may miss crossings.

For an up-and-out option, missing crossings overprices the option because some paths that should knock out remain alive.

We compare different time-step grids.

In [ ]:
def barrier_monitoring_frequency_experiment(
    config: ExoticMCConfig,
    barrier: float,
    step_grid: list[int],
    n_paths: int,
    seed: int = 777
) -> pd.DataFrame:
    """
    Compare discretely monitored barrier prices across monitoring grids.
    """
    rows = []

    for n_steps in step_grid:
        _, p = simulate_gbm_paths(
            config=config,
            n_paths=n_paths,
            n_steps=n_steps,
            seed=seed,
            antithetic=True
        )

        payoff_df = barrier_payoff_discrete(
            paths=p,
            strike=config.strike,
            barrier=barrier,
            option_type="call",
            barrier_type="up-and-out"
        )

        price_result = monte_carlo_price_from_payoff(
            payoff=payoff_df["barrier_payoff"].to_numpy(),
            maturity=config.maturity,
            risk_free_rate=config.risk_free_rate,
            label=f"up_and_out_call_{n_steps}_steps"
        )

        rows.append({
            "n_steps": n_steps,
            "price": price_result["price"],
            "standard_error": price_result["standard_error"],
            "knockout_rate": payoff_df["knocked_out_or_not_activated"].mean(),
            "average_payoff": payoff_df["barrier_payoff"].mean()
        })

    return pd.DataFrame(rows)


barrier_step_grid = [12, 26, 52, 126, 252, 504, 1008]

barrier_frequency_results = barrier_monitoring_frequency_experiment(
    config=config,
    barrier=barrier,
    step_grid=barrier_step_grid,
    n_paths=60_000,
    seed=888
)

barrier_frequency_results

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(barrier_frequency_results["n_steps"], barrier_frequency_results["price"], marker="o")
plt.xscale("log")
plt.title("Discrete Barrier Price vs Monitoring Frequency")
plt.xlabel("Number of monitoring steps")
plt.ylabel("Up-and-out call price")
plt.show()

## 13. Brownian-bridge correction for barrier crossing

For GBM, conditional on two endpoints, the log-price bridge can cross a barrier between steps.

For an up barrier $B$, if both endpoints are below $B$, the approximate crossing probability for log price is:

$$
\begin{aligned}
p_{\text{cross}} &= \exp \left( -\frac{2(\log B-\log S_t)(\log B-\log S_{t+\Delta t})} {\sigma^2\Delta t} \right)
\end{aligned}
$$
If either endpoint is already above the barrier, the crossing probability is 1.

For an up-and-out option, the survival probability across all intervals is approximately:

$$
\prod_i (1-p_{\text{cross},i})
$$
Then the payoff is weighted by survival probability.

This reduces the upward bias from missed barrier crossings.

In [ ]:
def up_barrier_survival_probability_bridge(
    paths: np.ndarray,
    barrier: float,
    volatility: float,
    maturity: float
) -> np.ndarray:
    """
    Approximate survival probability for continuously monitored up barrier
    using Brownian bridge crossing probabilities on log prices.
    """
    n_steps = paths.shape[1] - 1
    dt = maturity / n_steps

    log_paths = np.log(paths)
    log_barrier = log(barrier)

    left = log_paths[:, :-1]
    right = log_paths[:, 1:]

    # If either endpoint is above barrier, crossing occurred.
    endpoint_crossed = (left >= log_barrier) | (right >= log_barrier)

    distance_left = log_barrier - left
    distance_right = log_barrier - right

    crossing_prob = np.exp(
        -2.0 * distance_left * distance_right / (volatility ** 2 * dt)
    )

    crossing_prob = np.where(endpoint_crossed, 1.0, crossing_prob)
    crossing_prob = np.clip(crossing_prob, 0.0, 1.0)

    survival_prob = np.prod(1.0 - crossing_prob, axis=1)

    return survival_prob


def up_and_out_call_bridge_corrected(
    paths: np.ndarray,
    strike: float,
    barrier: float,
    volatility: float,
    maturity: float
) -> pd.DataFrame:
    """
    Brownian-bridge corrected up-and-out call payoff.
    """
    terminal = paths[:, -1]
    vanilla = np.maximum(terminal - strike, 0.0)
    survival_prob = up_barrier_survival_probability_bridge(
        paths=paths,
        barrier=barrier,
        volatility=volatility,
        maturity=maturity
    )

    corrected_payoff = vanilla * survival_prob

    return pd.DataFrame({
        "terminal": terminal,
        "vanilla_payoff": vanilla,
        "survival_probability": survival_prob,
        "bridge_corrected_payoff": corrected_payoff
    })


bridge_df = up_and_out_call_bridge_corrected(
    paths=barrier_paths,
    strike=config.strike,
    barrier=barrier,
    volatility=config.volatility,
    maturity=config.maturity
)

bridge_price = monte_carlo_price_from_payoff(
    payoff=bridge_df["bridge_corrected_payoff"].to_numpy(),
    maturity=config.maturity,
    risk_free_rate=config.risk_free_rate,
    label="up_and_out_call_bridge_corrected"
)

pd.DataFrame([up_and_out_call_discrete, bridge_price])

## 14. Lookback options

A floating-strike lookback call payoff is:

$$
S_T-\min_t S_t
$$
A floating-strike lookback put payoff is:

$$
\max_t S_t - S_T
$$
The payoff depends on the running minimum or maximum of the path.

Lookback options are sensitive to monitoring frequency because a coarse grid may miss true extrema.

In [ ]:
def lookback_payoff(
    paths: np.ndarray,
    option_type: str = "floating_call"
) -> np.ndarray:
    """
    Floating-strike lookback option payoff.
    """
    terminal = paths[:, -1]
    path_min = paths.min(axis=1)
    path_max = paths.max(axis=1)

    if option_type == "floating_call":
        return terminal - path_min

    if option_type == "floating_put":
        return path_max - terminal

    raise ValueError("option_type must be 'floating_call' or 'floating_put'.")


lookback_call_price = monte_carlo_price_from_payoff(
    payoff=lookback_payoff(barrier_paths, option_type="floating_call"),
    maturity=config.maturity,
    risk_free_rate=config.risk_free_rate,
    label="floating_lookback_call"
)

lookback_put_price = monte_carlo_price_from_payoff(
    payoff=lookback_payoff(barrier_paths, option_type="floating_put"),
    maturity=config.maturity,
    risk_free_rate=config.risk_free_rate,
    label="floating_lookback_put"
)

pd.DataFrame([lookback_call_price, lookback_put_price])

## 15. Lookback monitoring-frequency experiment

For a lookback call, increasing monitoring frequency gives the path more opportunities to observe a lower minimum.

That tends to increase the floating-strike lookback call value.

For a lookback put, finer monitoring gives more opportunities to observe a higher maximum.

In [ ]:
def lookback_monitoring_frequency_experiment(
    config: ExoticMCConfig,
    step_grid: list[int],
    n_paths: int,
    seed: int = 999
) -> pd.DataFrame:
    """
    Compare lookback prices across different monitoring frequencies.
    """
    rows = []

    for n_steps in step_grid:
        _, p = simulate_gbm_paths(
            config=config,
            n_paths=n_paths,
            n_steps=n_steps,
            seed=seed,
            antithetic=True
        )

        call_result = monte_carlo_price_from_payoff(
            payoff=lookback_payoff(p, option_type="floating_call"),
            maturity=config.maturity,
            risk_free_rate=config.risk_free_rate,
            label=f"floating_lookback_call_{n_steps}"
        )

        put_result = monte_carlo_price_from_payoff(
            payoff=lookback_payoff(p, option_type="floating_put"),
            maturity=config.maturity,
            risk_free_rate=config.risk_free_rate,
            label=f"floating_lookback_put_{n_steps}"
        )

        rows.append({
            "n_steps": n_steps,
            "floating_call_price": call_result["price"],
            "floating_call_se": call_result["standard_error"],
            "floating_put_price": put_result["price"],
            "floating_put_se": put_result["standard_error"]
        })

    return pd.DataFrame(rows)


lookback_frequency_results = lookback_monitoring_frequency_experiment(
    config=config,
    step_grid=[12, 26, 52, 126, 252, 504, 1008],
    n_paths=50_000,
    seed=1000
)

lookback_frequency_results

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(lookback_frequency_results["n_steps"], lookback_frequency_results["floating_call_price"], marker="o", label="Floating call")
plt.plot(lookback_frequency_results["n_steps"], lookback_frequency_results["floating_put_price"], marker="o", label="Floating put")
plt.xscale("log")
plt.title("Lookback Option Price vs Monitoring Frequency")
plt.xlabel("Number of monitoring steps")
plt.ylabel("Price")
plt.legend()
plt.show()

## 16. Convergence and standard error by exotic type

We compare convergence across several payoffs.

Different exotic payoffs can have different payoff variance, which affects Monte Carlo error.

A barrier option may have lower average payoff but complex discontinuity near the barrier.  
A lookback option may have high path-dependence and payoff dispersion.

In [ ]:
def price_exotic_suite(
    config: ExoticMCConfig,
    n_paths: int,
    n_steps: int,
    barrier: float,
    seed: int
) -> pd.DataFrame:
    """
    Price a suite of exotic payoffs using the same simulated paths.
    """
    _, p = simulate_gbm_paths(
        config=config,
        n_paths=n_paths,
        n_steps=n_steps,
        seed=seed,
        antithetic=True
    )

    asian = asian_payoffs(p, config.strike, option_type="call")
    barrier_disc = barrier_payoff_discrete(
        p,
        strike=config.strike,
        barrier=barrier,
        option_type="call",
        barrier_type="up-and-out"
    )
    barrier_bridge = up_and_out_call_bridge_corrected(
        p,
        strike=config.strike,
        barrier=barrier,
        volatility=config.volatility,
        maturity=config.maturity
    )

    results = [
        monte_carlo_price_from_payoff(
            asian["arithmetic_payoff"].to_numpy(),
            config.maturity,
            config.risk_free_rate,
            "arithmetic_asian_call"
        ),
        monte_carlo_price_from_payoff(
            barrier_disc["barrier_payoff"].to_numpy(),
            config.maturity,
            config.risk_free_rate,
            "up_and_out_call_discrete"
        ),
        monte_carlo_price_from_payoff(
            barrier_bridge["bridge_corrected_payoff"].to_numpy(),
            config.maturity,
            config.risk_free_rate,
            "up_and_out_call_bridge"
        ),
        monte_carlo_price_from_payoff(
            lookback_payoff(p, option_type="floating_call"),
            config.maturity,
            config.risk_free_rate,
            "floating_lookback_call"
        )
    ]

    return pd.DataFrame(results)


exotic_suite = price_exotic_suite(
    config=config,
    n_paths=config.n_paths,
    n_steps=config.n_steps,
    barrier=barrier,
    seed=1234
)

exotic_suite

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(exotic_suite["instrument"], exotic_suite["standard_error"])
plt.title("Monte Carlo Standard Error by Exotic Payoff")
plt.xlabel("Standard error")
plt.ylabel("Instrument")
plt.show()

## 17. Path diagnostics

For path-dependent options, it is useful to inspect path statistics:

- terminal price;
- path average;
- path maximum;
- path minimum;
- barrier crossing indicator.

This helps explain why exotic prices differ from vanilla prices.

In [ ]:
path_diagnostics = pd.DataFrame({
    "terminal": barrier_paths[:, -1],
    "path_average": barrier_paths[:, 1:].mean(axis=1),
    "path_min": barrier_paths.min(axis=1),
    "path_max": barrier_paths.max(axis=1),
})

path_diagnostics["up_barrier_crossed"] = path_diagnostics["path_max"] >= barrier

path_diagnostics.describe(percentiles=[0.01, 0.05, 0.50, 0.95, 0.99])

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(path_diagnostics["path_average"], bins=80, alpha=0.7, label="Path average")
plt.hist(path_diagnostics["terminal"], bins=80, alpha=0.5, label="Terminal")
plt.axvline(config.strike, linestyle="--", label="Strike")
plt.title("Path Average vs Terminal Price Distribution")
plt.xlabel("Price")
plt.ylabel("Count")
plt.legend()
plt.show()

## 18. Simple Greeks by bump-and-revalue

For exotic options, analytical Greeks are often unavailable.

A simple method is bump-and-revalue:

$$
\Delta \approx \frac{V(S_0+\epsilon)-V(S_0-\epsilon)}{2\epsilon}
$$
This is easy but noisy because two independent simulations add variance.

A better approach uses common random numbers: reuse the same normal shocks for both bumped simulations.

This notebook demonstrates the structure but keeps the implementation simple.

In [ ]:
def simulate_gbm_paths_from_shocks(
    config: ExoticMCConfig,
    shocks: np.ndarray,
    s0_override: float
) -> np.ndarray:
    """
    Simulate paths from a supplied shock matrix for common-random-number Greeks.
    """
    n_paths, n_steps = shocks.shape
    dt = config.maturity / n_steps

    log_increments = (
        (config.risk_free_rate - config.dividend_yield - 0.5 * config.volatility ** 2) * dt
        + config.volatility * sqrt(dt) * shocks
    )

    log_paths = np.zeros((n_paths, n_steps + 1))
    log_paths[:, 0] = log(s0_override)
    log_paths[:, 1:] = log(s0_override) + np.cumsum(log_increments, axis=1)

    return np.exp(log_paths)


def bump_and_revalue_delta_arithmetic_asian(
    config: ExoticMCConfig,
    bump: float,
    n_paths: int,
    n_steps: int,
    seed: int = 2029
) -> dict:
    """
    Estimate arithmetic Asian Delta using common random numbers.
    """
    local_rng = np.random.default_rng(seed)
    shocks = local_rng.standard_normal(size=(n_paths, n_steps))

    paths_up = simulate_gbm_paths_from_shocks(config, shocks, s0_override=config.s0 + bump)
    paths_down = simulate_gbm_paths_from_shocks(config, shocks, s0_override=config.s0 - bump)

    payoff_up = asian_payoffs(paths_up, config.strike, option_type="call")["arithmetic_payoff"].to_numpy()
    payoff_down = asian_payoffs(paths_down, config.strike, option_type="call")["arithmetic_payoff"].to_numpy()

    price_up = monte_carlo_price_from_payoff(
        payoff_up,
        config.maturity,
        config.risk_free_rate,
        "asian_up"
    )["price"]

    price_down = monte_carlo_price_from_payoff(
        payoff_down,
        config.maturity,
        config.risk_free_rate,
        "asian_down"
    )["price"]

    delta = (price_up - price_down) / (2 * bump)

    return {
        "price_up": price_up,
        "price_down": price_down,
        "bump": bump,
        "delta_estimate": delta,
        "n_paths": n_paths
    }


asian_delta_estimate = bump_and_revalue_delta_arithmetic_asian(
    config=config,
    bump=0.5,
    n_paths=60_000,
    n_steps=config.n_steps,
    seed=2029
)

pd.Series(asian_delta_estimate)

## 19. Saving exotic option outputs

The notebook saves:

1. vanilla validation prices;
2. Asian option results;
3. Asian control-variate result;
4. barrier monitoring-frequency results;
5. lookback monitoring-frequency results;
6. exotic suite comparison;
7. path diagnostics summary;
8. manifest with assumptions.

In [ ]:
output_dir = Path("data/processed/exotic_options_monte_carlo_v1")
output_dir.mkdir(parents=True, exist_ok=True)

vanilla_validation_path = output_dir / "vanilla_validation.csv"
asian_results_path = output_dir / "asian_option_results.csv"
barrier_frequency_path = output_dir / "barrier_monitoring_frequency.csv"
lookback_frequency_path = output_dir / "lookback_monitoring_frequency.csv"
exotic_suite_path = output_dir / "exotic_suite_prices.csv"
path_diagnostics_path = output_dir / "path_diagnostics_summary.csv"
greeks_path = output_dir / "asian_bump_and_revalue_delta.csv"
manifest_path = output_dir / "manifest.json"

vanilla_validation_df = pd.DataFrame([
    {**vanilla_call_mc, "bsm_reference": bsm_call_reference, "error_vs_bsm": vanilla_call_mc["price"] - bsm_call_reference},
    {**vanilla_put_mc, "bsm_reference": bsm_put_reference, "error_vs_bsm": vanilla_put_mc["price"] - bsm_put_reference},
])

asian_results_df = pd.DataFrame([
    asian_arithmetic_mc,
    asian_geometric_mc,
    {
        "instrument": "arithmetic_asian_call_control_variate",
        "price": asian_cv_result["control_variate_price"],
        "standard_error": asian_cv_result["control_variate_standard_error"],
        "lower_95": asian_cv_result["control_variate_price"] - 1.96 * asian_cv_result["control_variate_standard_error"],
        "upper_95": asian_cv_result["control_variate_price"] + 1.96 * asian_cv_result["control_variate_standard_error"],
        "n_paths": config.n_paths,
        "variance_reduction_ratio": asian_cv_result["variance_reduction_ratio"]
    }
])

vanilla_validation_df.to_csv(vanilla_validation_path, index=False)
asian_results_df.to_csv(asian_results_path, index=False)
barrier_frequency_results.to_csv(barrier_frequency_path, index=False)
lookback_frequency_results.to_csv(lookback_frequency_path, index=False)
exotic_suite.to_csv(exotic_suite_path, index=False)
path_diagnostics.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).to_csv(path_diagnostics_path)
pd.DataFrame([asian_delta_estimate]).to_csv(greeks_path, index=False)

manifest = {
    "dataset_name": "exotic_options_monte_carlo_outputs",
    "schema_version": "exotic_options_monte_carlo_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "barrier": barrier,
    "bsm_call_reference": float(bsm_call_reference),
    "bsm_put_reference": float(bsm_put_reference),
    "geometric_asian_closed_form": float(geometric_asian_cf),
    "asian_control_variate_variance_reduction_ratio": float(asian_cv_result["variance_reduction_ratio"]),
    "notes": [
        "Paths are simulated under risk-neutral GBM using exact discrete transitions.",
        "Asian arithmetic option uses Monte Carlo; geometric Asian is used as a control variate.",
        "Barrier option demonstrates monitoring-frequency bias.",
        "Brownian bridge correction approximates continuously monitored up-barrier crossing.",
        "Lookback options are discretely monitored on the simulated grid."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

vanilla_validation_path, asian_results_path, barrier_frequency_path, lookback_frequency_path, manifest_path

## 20. Practical checklist for exotic Monte Carlo pricing

Before trusting an exotic option Monte Carlo price, check:

1. **Measure**  
   Are paths simulated under the risk-neutral measure?

2. **Vanilla validation**  
   Does the same engine reproduce BSM vanilla prices?

3. **Payoff definition**  
   Is the payoff discretely or continuously monitored?

4. **Monitoring grid**  
   Is the time grid fine enough for the payoff?

5. **Barrier crossings**  
   Are crossings between grid points missed? Is Brownian-bridge correction needed?

6. **Confidence interval**  
   Is standard error reported?

7. **Variance reduction**  
   Are antithetic or control variates available?

8. **Random seed**  
   Is the run reproducible?

9. **Memory footprint**  
   Can the path matrix fit in memory?

10. **Greeks**  
   Are Greeks estimated with common random numbers or a more stable method?

11. **Model risk**  
   Is GBM sufficient, or do jumps/stochastic volatility matter?

12. **Audit trail**  
   Are assumptions, monitoring conventions, and outputs saved?

## 21. Limitations

### 21.1 GBM is a simplified process

The notebook assumes constant volatility and continuous paths.

Real markets have:

- stochastic volatility;
- jumps;
- volatility smiles;
- rough volatility;
- liquidity effects;
- transaction costs.

### 21.2 Discrete monitoring can bias prices

Barrier and lookback options can be very sensitive to monitoring frequency.

A daily-monitored barrier is not the same as a continuously monitored barrier.

### 21.3 Brownian-bridge correction is model-specific

The bridge formula relies on GBM log-price dynamics between monitoring dates.

It may not apply directly under jumps, stochastic volatility, or discrete price limits.

### 21.4 Monte Carlo convergence is slow

Error decreases at:

$$
O(M^{-1/2})
$$
So high accuracy can be expensive.

### 21.5 Greeks can be noisy

Bump-and-revalue Greeks require careful variance reduction.

Pathwise and likelihood-ratio methods may be preferable for some payoffs.

### 21.6 Memory usage can be large

Full path simulation stores a matrix of size:

$$
M \times (N+1)
$$
This can become expensive for many paths, fine grids, or multi-asset exotics.

### 21.7 American-style exotics are harder

This notebook prices European-style path-dependent payoffs.

Early exercise requires additional methods such as Longstaff-Schwartz least-squares Monte Carlo.

## 22. Research frontier and current directions

Exotic option Monte Carlo remains an active area because realistic payoffs, models, and markets are complex.

### 22.1 Longstaff-Schwartz least-squares Monte Carlo

American and Bermudan options require optimal stopping.

The Longstaff-Schwartz method estimates continuation values by regression on simulated paths.

A future notebook could price a Bermudan put and compare exercise policies across basis functions.

### 22.2 Multilevel Monte Carlo

Path-dependent options can be expensive because fine time grids are required.

Multilevel Monte Carlo combines many cheap coarse paths with fewer expensive fine paths to reduce computational cost.

A future notebook could apply MLMC to Asian or barrier options and compare error versus runtime.

### 22.3 Brownian bridge and conditional Monte Carlo

Barrier options are highly sensitive to missed crossings.

Brownian-bridge methods reduce monitoring bias by integrating over the unobserved path between grid points.

A future notebook could compare naive discrete barriers, Brownian-bridge correction, and analytical barrier formulas.

### 22.4 Monte Carlo Greeks

Exotic Greeks can be estimated using:

- bump-and-revalue;
- pathwise derivatives;
- likelihood-ratio methods;
- Malliavin calculus;
- automatic differentiation;
- adjoint methods.

A future notebook could compare Delta estimators for Asian and barrier options.

### 22.5 GPU and differentiable Monte Carlo

Monte Carlo is parallel and can benefit from GPUs.

Differentiable frameworks such as JAX or PyTorch can also compute sensitivities through simulation.

A future notebook could implement an Asian option pricer in JAX and compute Delta automatically.

### 22.6 Exotics under stochastic volatility and jumps

Real exotic books are often sensitive to volatility smile dynamics and jump risk.

A future notebook could price barriers under Heston, Merton jump diffusion, and Bates models to show model-risk differences.

### 22.7 Neural pricing surrogates

Neural networks can approximate exotic option prices across parameter grids.

A future notebook could train a neural surrogate model on Monte Carlo prices and test interpolation across strikes, maturities, and barriers.

## 23. Suggested follow-up notebooks

This notebook naturally leads to:

1. **02_08_dynamic_delta_hedging_simulation**  
   Testing hedging errors for path-dependent and vanilla options.

2. **02_13_multilevel_monte_carlo_pricing**  
   Reducing cost for path-dependent Monte Carlo.

3. **02_11_heston_model_calibration**  
   Moving from GBM to stochastic volatility.

4. **02_Bates_stochastic_volatility_with_jumps**  
   Pricing exotics under stochastic volatility plus jumps.

5. **04_06_var_cvar_and_stress_testing**  
   Using path simulations for nonlinear portfolio risk.

6. **06_11_cpp_python_bindings_pybind11**  
   Moving Monte Carlo path kernels into compiled code.

7. **07_02_volatility_surface_pricing_and_alpha**  
   Connecting exotic pricing to smile and model-risk assumptions.

## 24. Summary

This notebook priced exotic options using Monte Carlo simulation.

It showed how to:

1. validate a Monte Carlo path engine against BSM vanilla prices;
2. price arithmetic and geometric Asian options;
3. use geometric Asian closed form as a control variate;
4. price discretely monitored barrier options;
5. show monitoring-frequency bias;
6. apply Brownian-bridge correction for up-barrier crossing;
7. price floating-strike lookback options;
8. compare standard errors across exotic payoffs;
9. estimate a simple bump-and-revalue Delta using common random numbers;
10. save pricing outputs and diagnostics.

The key computational takeaway is:

> Path-dependent Monte Carlo requires more than simulating paths. Monitoring conventions, variance reduction, confidence intervals, and diagnostics are part of the pricing engine.

The key financial takeaway is:

> Exotic option prices are highly sensitive to path features such as averages, extrema, and barrier crossings. The model and monitoring convention can matter as much as the payoff formula.

## 25. Further reading

### Monte Carlo and exotic options

- Glasserman, P. *Monte Carlo Methods in Financial Engineering.*
- Jäckel, P. *Monte Carlo Methods in Finance.*
- Hull, J. C. *Options, Futures, and Other Derivatives.*
- Shreve, S. E. *Stochastic Calculus for Finance II.*

### Exotic option topics

- Asian option pricing.
- Barrier option monitoring correction.
- Brownian bridge methods.
- Lookback option pricing.
- Longstaff-Schwartz least-squares Monte Carlo.

### Future extensions

- Multilevel Monte Carlo for exotics.
- Monte Carlo Greeks.
- JAX/CUDA Monte Carlo.
- Heston and jump-diffusion exotics.
- Neural pricing surrogates.
- Bermudan and American option simulation.